# Acceptance: float64 by default

**This is a sign-off page, not a tutorial.** It checks that importing summer4 turns
on JAX's 64-bit mode, so models solve in float64 as they did in summer2, and that
float32 is still one line away (plan `plans/float64-default.plan.md`).

JAX computes in float32 unless its process-wide `jax_enable_x64` flag is on.
summer4 now turns that flag on when it is imported, unless the `JAX_ENABLE_X64`
environment variable is set, in which case JAX has already applied your choice.

The model is deliberately hard for float32: an SIR epidemic in **100 million**
people, seeded with **one** infectious person. Near 1e8, float32 can only
represent every 8th whole number, so the susceptible compartment cannot record
the first few infections at all.

To compare the two widths in one notebook, the helper below switches the flag
before compiling and running, then puts it back. In your own work you choose
once, before compiling. Nothing here asserts; the automated checks live in
`tests/test_float64_default.py`. Run every cell top to bottom (a few seconds).
Each section ends with a **What to check** box.

| | You are checking that… |
|---|---|
| A | after `import summer4` the flag is on and results are float64 |
| B | float64 records small changes to a large compartment; float32 loses them |
| C | float64 conserves the population; float32 drifts by hundreds of people |
| D | float32 is still available, and what it costs and saves here |

In [ ]:
import time

import jax
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import FlowModel, Param, Property, PropertyMap, TransitionFlow
from summer4.epi import ForceOfInfection

state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
y0 = np.array([1e8 - 1.0, 1.0, 0.0])
params = {"contact_rate": 0.15, "recovery_rate": 0.1}
# An adaptive solver, so its own error is far below the effects of the number width.
run_kwargs = dict(t0=0.0, t1=365.0, dt=1.0, solver="tsit5")


def compile_model():
    """An SIR model with homogeneous mixing."""
    model = FlowModel(pmap)
    foi = ForceOfInfection("foi", infectious=state["I"], contact_rate=Param("contact_rate"))
    model.add_flow(TransitionFlow("infection", state["S"], state["I"], foi))
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery_rate")))
    return model.compile()


def run_at(x64):
    """Compile and run with jax_enable_x64 set to ``x64``, then restore the flag."""
    previous = jax.config.jax_enable_x64
    jax.config.update("jax_enable_x64", x64)
    try:
        return compile_model().run(params, y0, **run_kwargs)
    finally:
        jax.config.update("jax_enable_x64", previous)


def compartments(result):
    """Compartment sizes over time, as float64 for comparison."""
    values = np.asarray(result["compartments"].values.data).astype(np.float64)
    days = np.asarray(result["compartments"].times.values)
    return pd.DataFrame(values, columns=list(state.traits), index=pd.Index(days, name="day"))


result64 = run_at(True)
result32 = run_at(False)
c64 = compartments(result64)
c32 = compartments(result32)

## A. The default after `import summer4`

The first table shows JAX's flag in this kernel, which nobody set by hand, and the
dtype of the compartments from a default run (`result64`) and from the float32
run (`result32`).

The chart shows why the width matters for this model: the gap between
neighbouring representable numbers at 100 million. Changes smaller than half
that gap are lost when they are added to a compartment that size.

In [ ]:
pd.DataFrame(
    {
        "jax_enable_x64": [jax.config.jax_enable_x64, False],
        "compartments dtype": [
            str(result64["compartments"].values.data.dtype),
            str(result32["compartments"].values.data.dtype),
        ],
    },
    index=pd.Index(["default (after import summer4)", "opted out"], name="run"),
)

In [ ]:
spacing = pd.Series(
    {"float64": np.spacing(np.float64(1e8)), "float32": float(np.spacing(np.float32(1e8)))},
    name="gap between neighbouring numbers at 1e8",
)
fig = spacing.plot.bar(
    log_y=True,
    title="Smallest step a compartment of 100 million can take",
    labels={"index": "width", "value": "people (log scale)"},
)
fig.update_layout(showlegend=False)

> **What to check (A).** The flag is `True` and the default run is `float64`
> without any setup in this notebook. The bars are about 1.5e-8 people for
> float64 and 8 people for float32.

## B. Small changes to a large compartment

People often read cumulative infections off the susceptible compartment,
`S(0) − S(t)`. The chart plots that for both widths over the first 100 days,
against infections integrated from the infection flow: its rate summed across
compartments with `sum_by()`, integrated over each day with
`integrate_intervals()`, then accumulated with `cumulative()`. That is accurate
at both widths, because the rate is computed from the small infectious
compartment.

In [ ]:
per_day = result64["infection"].sum_by().integrate_intervals().cumsum()
from_flow = pd.Series(np.asarray(per_day.values), index=np.asarray(per_day.times.values))
early = pd.DataFrame(
    {
        "S(0) − S(t), float64": y0[0] - c64["S"],
        "S(0) − S(t), float32": y0[0] - c32["S"],
        "integrated infection flow": from_flow,
    },
    index=c64.index,
).fillna({"integrated infection flow": 0.0}).loc[:100]
early.plot(
    title="Infections so far, read from the susceptible compartment",
    labels={"value": "people", "variable": "series"},
)

In [ ]:
early.loc[[30, 60, 90]].round(1)

> **What to check (B).** The float64 line sits on the integrated flow (the
> table agrees to a fraction of a person). The float32 line moves in steps of 8,
> is at or below zero for about two months (S cannot drop below 100 million while
> each day's infections are smaller than the step), and is still far below
> float64 at day 90.

## C. Conservation of the population

The model has no births or deaths, so `S + I + R` should stay at 100 million. The
chart plots how far the total strays over the year, on a log scale. Points
where the total is exactly 100 million cannot be drawn on a log scale and are
left out.

In [ ]:
drift = pd.DataFrame(
    {
        "float64": (c64.sum(axis=1) - 1e8).abs(),
        "float32": (c32.sum(axis=1) - 1e8).abs(),
    }
).replace(0.0, np.nan)
drift.plot(
    log_y=True,
    title="Change in total population, 100 million people",
    labels={"value": "|N(t) − N(0)|, people (log scale)", "variable": "width"},
)

In [ ]:
pd.DataFrame(
    {
        "largest drift": drift.max(),
        "recovered at day 365": [c64["R"].iloc[-1], c32["R"].iloc[-1]],
    }
).round(2)

> **What to check (C).** float64 drifts by less than a millionth of a person;
> float32 by hundreds of people. The recovered totals at day 365 differ by tens
> of people too.

## D. Choosing float32

float32 is still available. Either set the environment variable before starting
Python or Jupyter,

```bash
JAX_ENABLE_X64=0 pixi run notebook
```

or turn the flag off after importing summer4 and before compiling a model:

```python
import summer4
import jax

jax.config.update("jax_enable_x64", False)
```

That is what `run_at(False)` does above. The flag is process-wide, so it also
applies to any other JAX code in the same process.

The chart compares how long one run takes at each width on this machine (the
median of 20 runs after compiling). On a CPU, and for a model this size, the
difference is small; float64 costs more on most GPUs, and some, such as Apple's
Metal backend, do not support it.

In [ ]:
def median_run_seconds(x64, repeats=20):
    previous = jax.config.jax_enable_x64
    jax.config.update("jax_enable_x64", x64)
    try:
        model = compile_model()
        model.run(params, y0, **run_kwargs)  # compile once
        times = []
        for _ in range(repeats):
            start = time.perf_counter()
            model.run(params, y0, **run_kwargs)["compartments"].values.data.block_until_ready()
            times.append(time.perf_counter() - start)
        return float(np.median(times))
    finally:
        jax.config.update("jax_enable_x64", previous)


timing = pd.Series(
    {"float64": median_run_seconds(True), "float32": median_run_seconds(False)},
    name="median seconds per run",
)
fig = timing.plot.bar(
    title="Time per run, one year with tsit5",
    labels={"index": "width", "value": "seconds"},
)
fig.update_layout(showlegend=False)

In [ ]:
jax.config.jax_enable_x64

> **What to check (D).** The bars are close (timings vary between runs and
> machines). The last cell prints `True`: the comparisons on this page put the
> flag back each time.